In [ ]:
import xgboost as xgb


In [13]:
import pandas as pd

In [15]:
from sklearn.feature_extraction import DictVectorizer

In [25]:
from sklearn.metrics import root_mean_squared_error

In [10]:

import mlflow
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('nyc_taxi_experiment')

<Experiment: artifact_location='/Users/matthiasmotl/neuefische/repositories/dtc/dtc-mlops/02_experiment_tracking/mlruns/1', creation_time=1747236415301, experiment_id='1', last_update_time=1747236415301, lifecycle_stage='active', name='nyc_taxi_experiment', tags={}>

In [2]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

Okay, let's break down this Python import statement:

`from hyperopt import fmin, tpe, hp, STATUS_OK, Trials`

This line is importing specific components from the `hyperopt` library. `hyperopt` is a popular Python library for **hyperparameter optimization**, particularly for machine learning models. It helps you find the best set of hyperparameters for your model more efficiently than manual tuning, grid search, or random search.

Here's what each imported component does:

1.  **`hyperopt` (the library itself):**
    *   **Purpose:** To perform distributed, asynchronous hyperparameter optimization. It allows you to define a search space for your model's hyperparameters and then uses various algorithms to search that space for the combination that yields the best performance (e.g., lowest loss, highest accuracy).

2.  **`fmin`:**
    *   **Purpose:** This is the **core function** in `hyperopt` that actually **runs the optimization process**.
    *   **Stands for:** "Function MINimization".
    *   **How it works:** You provide `fmin` with:
        *   An **objective function** (a Python function you define that takes hyperparameters as input and returns a score/loss to be minimized).
        *   A **search space** (defined using `hp`).
        *   An **optimization algorithm** (like `tpe`).
        *   The maximum number of evaluations to perform.
    *   **Output:** It returns the set of hyperparameters that resulted in the best score (lowest loss) found during the optimization.

3.  **`tpe`:**
    *   **Purpose:** This is one of the **optimization algorithms** that `fmin` can use to search the hyperparameter space.
    *   **Stands for:** "Tree-structured Parzen Estimator".
    *   **How it works:** `tpe` is a Bayesian optimization algorithm. It intelligently selects the next set of hyperparameters to try based on the results of previous trials. It builds a probabilistic model of the objective function and uses it to decide which hyperparameters are most promising to evaluate next. It's generally more efficient than random search or grid search.
    *   **Usage:** You pass `tpe.suggest` as the `algo` argument to `fmin`.

4.  **`hp`:**
    *   **Purpose:** This module is used to **define the search space** for your hyperparameters.
    *   **Stands for:** "Hyperparameter" (or "Hyperopt Parameters").
    *   **How it works:** `hp` provides various functions to specify different types of distributions and ranges for your hyperparameters:
        *   `hp.choice(label, options)`: Choose one from a list of options (e.g., `hp.choice('optimizer', ['adam', 'sgd'])`).
        *   `hp.uniform(label, low, high)`: A continuous uniform distribution between `low` and `high` (e.g., `hp.uniform('learning_rate', 0.0001, 0.1)`).
        *   `hp.quniform(label, low, high, q)`: A quantized uniform distribution (e.g., for integers like number of layers: `hp.quniform('num_layers', 1, 5, 1)`).
        *   `hp.loguniform(label, low, high)`: A log-uniform distribution (useful for parameters that vary over several orders of magnitude, like learning rates).
        *   And others like `hp.normal`, `hp.lognormal`, etc.
    *   **Usage:** You create a dictionary where keys are hyperparameter names (labels) and values are these `hp` expressions. This dictionary is passed as the `space` argument to `fmin`.

5.  **`STATUS_OK`:**
    *   **Purpose:** This is a **constant string** (`'ok'`) used within your objective function.
    *   **How it works:** Your objective function (the one `fmin` tries to minimize) needs to return a dictionary. This dictionary must include a `'status'` key.
    *   **Usage:** If the evaluation of your model with a given set of hyperparameters was successful, you set `{'status': STATUS_OK}` in the return dictionary. This tells `hyperopt` that the trial completed without errors. Other statuses like `STATUS_FAIL` can also be used. The dictionary also typically includes a `'loss'` key with the value to be minimized.

6.  **`Trials`:**
    *   **Purpose:** This is a **class** used to **store information about each iteration (trial)** of the optimization process.
    *   **How it works:** You create an instance of the `Trials` class and pass it to the `fmin` function (via the `trials` argument). `fmin` will then populate this object with details about each evaluation: the hyperparameters tried, the resulting loss, status, and other metadata.
    *   **Usage:**
        *   **Inspection:** You can inspect the `trials.results`, `trials.losses()`, `trials.statuses()` etc., after the optimization to see how it progressed.
        *   **Resuming:** It can be used to resume an interrupted optimization run.
        *   **Analysis:** Helps in analyzing the search process and understanding which hyperparameter ranges were more promising.

**In Summary:**

You're importing these specific tools from `hyperopt` to:

1.  Define a **search space** for your hyperparameters using `hp`.
2.  Write an **objective function** that evaluates a set of hyperparameters and returns a loss, indicating success with `STATUS_OK`.
3.  Use the `fmin` function to run the optimization, telling it to use the `tpe` algorithm.
4.  Keep track of all the attempts and their results using a `Trials` object.

This setup allows you to systematically and efficiently find good hyperparameter configurations for your models.

In [3]:
from hyperopt.pyll import scope

Okay, let's break down `from hyperopt.pyll import scope`.

This import statement is bringing the `scope` object/module into your current Python namespace from the `hyperopt.pyll` submodule.

1.  **`hyperopt` (The main library):**
    *   As we discussed before, `hyperopt` is for hyperparameter optimization.

2.  **`pyll` (Submodule of `hyperopt`):**
    *   **Stands for:** "Python Language for L-systems" (though the L-system part isn't crucial for a basic understanding of its use in `hyperopt`).
    *   **Purpose:** `pyll` is the core symbolic computation graph library that `hyperopt` uses under the hood to define and manipulate the search space.
    *   When you use `hp` functions like `hp.uniform('x', 0, 1)`, you're not getting a random number immediately. Instead, you're creating a `pyll` "apply node" or a "graph node" that *represents* the operation of sampling from a uniform distribution.
    *   `hyperopt`'s `fmin` function takes this `pyll` graph (your search space) and evaluates it repeatedly (sampling actual values) to find the best hyperparameters.

3.  **`scope` (Object/Namespace within `pyll`):**
    *   **Purpose:** `scope` provides a way to incorporate standard Python functions and some predefined utility functions directly into your `pyll` search space definitions. It acts as a bridge, allowing certain operations to be part of the symbolic graph.
    *   **How it works:**
        *   **Namespace for Predefined Functions:** `scope` contains a collection of helpful functions that can operate on `pyll` graph nodes. The most common one you'll see is `scope.int()`.
        *   **Decorator for Custom Functions:** You can also use `@scope.define` as a decorator to register your own Python functions, making them available to be used within a `pyll` graph.

**Why is `scope` needed?**

When you define a search space with `hp` functions, these functions return `pyll` graph nodes. If you want to perform an operation on the *potential value* represented by that node *before* it's passed to your objective function, you need an operation that `pyll` understands.

**Common Use Cases for `scope`:**

1.  **Type Casting (Most Common - `scope.int`):**
    Many `hp` functions like `hp.quniform` (quantized uniform) or `hp.uniform` return float-like values. However, many machine learning parameters require integers (e.g., number of trees in a random forest, number of neurons in a layer, batch size).
    `scope.int()` allows you to ensure that the value sampled for a hyperparameter is cast to an integer *within the `pyll` graph itself*.

    ```python
    from hyperopt import hp
    from hyperopt.pyll import scope

    search_space = {
        # 'n_estimators' will be a float from quniform, e.g., 50.0, 51.0
        # We need an integer for many libraries (like scikit-learn's RandomForest)
        'n_estimators': scope.int(hp.quniform('n_estimators_float', 50, 200, 1)),
        'max_depth': scope.int(hp.quniform('max_depth_float', 3, 15, 1))
    }
    ```
    In this example:
    *   `hp.quniform('n_estimators_float', 50, 200, 1)` creates a `pyll` node that will sample values like 50.0, 51.0, ..., 200.0.
    *   `scope.int(...)` wraps this node, creating another `pyll` node that represents the operation of taking the integer part of the sampled value. So, `fmin` will ultimately pass an integer (e.g., 50, 51) to your objective function for `'n_estimators'`.

2.  **Applying Simple Mathematical Functions:**
    `scope` includes other utility functions like `scope.power`, `scope.minimum`, `scope.maximum`, `scope.log`, `scope.exp`, etc., that can be used to transform hyperparameter values within the search space definition.

    ```python
    from hyperopt import hp
    from hyperopt.pyll import scope
    import numpy as np # For np.log10, np.power

    search_space = {
        # Define learning rate on a log scale, then convert it back
        'learning_rate': scope.power(10, hp.uniform('log_lr', -5, -1))
        # 'log_lr' will be sampled from -5 to -1
        # 'learning_rate' will then be 10^log_lr
    }
    ```

3.  **Using Custom Functions (Advanced with `@scope.define`):**
    If you have a more complex transformation, you can define your own Python function and make it available to `pyll` using `@scope.define`.

    ```python
    from hyperopt import hp
    from hyperopt.pyll import scope

    @scope.define
    def custom_transform(x, y):
        return (x + y) / 2

    search_space = {
        'param_a': hp.uniform('a', 0, 1),
        'param_b': hp.uniform('b', 1, 2),
        'combined': custom_transform(hp.qnormal('a_ref', mu=0, sigma=1, q=1), hp.uniform('b_ref', 0,1))
        # Note: For 'combined' to work like this, 'a_ref' and 'b_ref' would typically be
        #       defined using hp.* and then referenced by their labels.
        # A more direct pyll way:
        # 'combined': custom_transform(hp.uniform('x_node',0,1), hp.uniform('y_node',0,1))
    }
    ```
    This is less common for typical hyperparameter tuning but shows the extensibility.

**In Summary:**

`from hyperopt.pyll import scope` imports a utility that allows you to:

*   **Access predefined functions** (like `scope.int`, `scope.power`) that can operate on `hyperopt` search space nodes (`hp.*` expressions).
*   **Extend `pyll`'s capabilities** by defining your own functions that can be used within the symbolic search space graph.

The most frequent reason you'll see it is for `scope.int()` to ensure hyperparameters that need to be integers are correctly cast before being passed to your objective function.


In [18]:
def read_dataframe(filename):
    if filename.endswith('.csv'):
        df = pd.read_csv(filename)

        df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
        df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    elif filename.endswith('.parquet'):
        df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [19]:
df_train = read_dataframe('../data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('../data/green_tripdata_2021-02.parquet')

In [5]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

NameError: name 'df_train' is not defined

In [ ]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [ ]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [ ]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

Okay, let's break down these two lines of Python code, which are very common when working with the XGBoost library:

```python
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)
```

Both lines are doing essentially the same thing but for different datasets (training and validation).

**Core Concept: `xgb.DMatrix`**

*   **`xgb`:** This is the alias typically used when you import the XGBoost library (e.g., `import xgboost as xgb`).
*   **`DMatrix`:** This is a specialized internal data structure used by XGBoost. It's designed to be highly optimized for both memory efficiency and training speed when working with XGBoost's algorithms.

**Breaking Down the First Line: `train = xgb.DMatrix(X_train, label=y_train)`**

1.  **`X_train`**:
    *   This variable is expected to contain your **feature data** for the **training set**.
    *   It's typically a 2D array-like structure where:
        *   Rows represent individual samples (e.g., individual customers, images, data points).
        *   Columns represent different features or attributes of those samples (e.g., age, income, pixel values).
    *   Common data types for `X_train` include:
        *   NumPy arrays (`numpy.ndarray`)
        *   Pandas DataFrames (`pandas.DataFrame`)
        *   SciPy sparse matrices (e.g., `scipy.sparse.csr_matrix`, useful for high-dimensional, sparse data)

2.  **`label=y_train`**:
    *   **`label`**: This is a keyword argument to the `DMatrix` constructor. It tells XGBoost that the data being passed next (`y_train`) represents the target variable or "labels."
    *   **`y_train`**: This variable is expected to contain your **target variable** (the values you want to predict) for the **training set**.
    *   It's typically a 1D array-like structure with the same number of elements as there are rows in `X_train`. Each element in `y_train` corresponds to the label for the respective sample in `X_train`.
    *   Common data types for `y_train` include:
        *   NumPy arrays
        *   Pandas Series

3.  **`xgb.DMatrix(X_train, label=y_train)`**:
    *   This part calls the `DMatrix` constructor.
    *   It takes your raw training features (`X_train`) and training labels (`y_train`) and converts them into XGBoost's optimized internal format.
    *   This process might involve:
        *   Checking data types.
        *   Handling missing values (XGBoost has built-in ways to handle NaNs).
        *   Optimizing the data layout for faster access during training.

4.  **`train = ...`**:
    *   The newly created `DMatrix` object, which now holds your training data in an XGBoost-friendly format, is assigned to the variable `train`.
    *   This `train` object will be passed to XGBoost's training functions (e.g., `xgb.train()` or the `fit` method of scikit-learn wrapper).

**Breaking Down the Second Line: `valid = xgb.DMatrix(X_val, label=y_val)`**

This line is analogous to the first, but it prepares your **validation dataset**.

1.  **`X_val`**:
    *   Your **feature data** for the **validation set**. This dataset is not used for training the model's parameters directly but is used to monitor performance during training (e.g., for early stopping) or to evaluate the model after training on data it hasn't seen before.
    *   It should have the same features (columns) as `X_train`.

2.  **`label=y_val`**:
    *   **`y_val`**: Your **target variable** for the **validation set**.

3.  **`xgb.DMatrix(X_val, label=y_val)`**:
    *   Creates an XGBoost `DMatrix` object from your validation features and labels.

4.  **`valid = ...`**:
    *   The resulting `DMatrix` object for the validation set is assigned to the variable `valid`.
    *   This `valid` object is often used in the `evals` parameter of `xgb.train()` to monitor metrics like accuracy or logloss on the validation set as the model trains.

**Why use `DMatrix`?**

*   **Performance:** XGBoost's algorithms are written in C++ and are highly optimized. The `DMatrix` format allows for efficient data transfer and processing by these underlying algorithms, leading to faster training times and lower memory usage compared to using raw Python objects like NumPy arrays directly in every iteration.
*   **Feature Handling:** `DMatrix` can handle various aspects like feature names, missing values, and sample weights in a consistent way.
*   **Interface:** It provides a standardized way to feed data into XGBoost models.

**In summary:**

These two lines are crucial preprocessing steps when using XGBoost. They convert your raw training and validation datasets (features and labels) into `DMatrix` objects, which is XGBoost's preferred internal data format for optimal performance and efficiency.

In [ ]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)#, squared=False)
        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

In [ ]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0), #exp(-3) to exp(0) - 0.05 to 1
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': 42
}

In [ ]:
best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials())

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

/Users/matthiasmotl/opt/anaconda3/envs/experiment_tracking/lib/python3.9/site-packages/xgboost/core.py:160: UserWarning: [13:47:45] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1703076406455/work/src/objective/regression_obj.cu:209: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:7.96755                           
[1]	validation-rmse:6.92250                           
[2]	validation-rmse:6.66663                           
[3]	validation-rmse:6.58893                           
[4]	validation-rmse:6.55667                           
[5]	validation-rmse:6.53926                           
[6]	validation-rmse:6.53121                           
[7]	validation-rmse:6.52666                           
[8]	validation-rmse:6.51877                           
[9]	validation-rmse:6.51285                           
[10]	validation-rmse:6.51029                          
[11]	validation-rmse:6.50712                          
[12]	validation-rmse:6.49800                          
[13]	validation-rmse:6.49451                          
[14]	validation-rmse:6.49031                          
[15]	validation-rmse:6.48694                          
[16]	validation-rmse:6.48239                          
[17]	validation-rmse:6.47976                          
[18]	valid

/Users/matthiasmotl/opt/anaconda3/envs/experiment_tracking/lib/python3.9/site-packages/xgboost/core.py:160: UserWarning: [13:48:05] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1703076406455/work/src/objective/regression_obj.cu:209: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:11.27558                                                   
[1]	validation-rmse:10.48244                                                   
[2]	validation-rmse:9.81340                                                    
[3]	validation-rmse:9.25511                                                    
[4]	validation-rmse:8.78901                                                    
[5]	validation-rmse:8.40420                                                    
[6]	validation-rmse:8.08489                                                    
[7]	validation-rmse:7.82268                                                    
[8]	validation-rmse:7.61003                                                    
[9]	validation-rmse:7.43610                                                    
[10]	validation-rmse:7.29481                                                   
[11]	validation-rmse:7.17557                                                   
[12]	validation-rmse:7.08027            

...

[0]	validation-rmse:11.51547                                                    
[1]	validation-rmse:10.89600                                                    
[2]	validation-rmse:10.34726                                                    
[3]	validation-rmse:9.86342                                                     
[4]	validation-rmse:9.43716                                                     
[5]	validation-rmse:9.06230                                                     
[6]	validation-rmse:8.73494                                                     
[7]	validation-rmse:8.44864                                                     
[8]	validation-rmse:8.20001                                                     
[9]	validation-rmse:7.98264                                                     
[10]	validation-rmse:7.79522                                                    
[11]	validation-rmse:7.63132                                                    
[12]	validation-rmse:7.49031

/Users/matthiasmotl/opt/anaconda3/envs/experiment_tracking/lib/python3.9/site-packages/xgboost/core.py:160: UserWarning: [14:43:54] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1703076406455/work/src/objective/regression_obj.cu:209: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:10.13290                                                    
[1]	validation-rmse:8.78925                                                     
[2]	validation-rmse:7.94144                                                     
[3]	validation-rmse:7.41875                                                     
[4]	validation-rmse:7.09638                                                     
[5]	validation-rmse:6.90220                                                     
[6]	validation-rmse:6.78325                                                     
[7]	validation-rmse:6.70544                                                     
[8]	validation-rmse:6.65349                                                     
[9]	validation-rmse:6.61140                                                     
[10]	validation-rmse:6.58654                                                    
[11]	validation-rmse:6.56307                                                    
[12]	validation-rmse:6.54825

/Users/matthiasmotl/opt/anaconda3/envs/experiment_tracking/lib/python3.9/site-packages/xgboost/core.py:160: UserWarning: [14:44:35] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1703076406455/work/src/objective/regression_obj.cu:209: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:10.94061                                                    
[1]	validation-rmse:9.94067                                                     
[2]	validation-rmse:9.15763                                                     
[3]	validation-rmse:8.54104                                                     
[4]	validation-rmse:8.08487                                                     
[5]	validation-rmse:7.71737                                                     
[6]	validation-rmse:7.44764                                                     
[7]	validation-rmse:7.24629                                                     
[8]	validation-rmse:7.08377                                                     
[9]	validation-rmse:6.96264                                                     
[10]	validation-rmse:6.87161                                                    
[11]	validation-rmse:6.80527                                                    
[12]	validation-rmse:6.75324